In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// --- ИСПОЛНЯЕМАЯ ЧАСТЬ ---

// Создаем список животных
List<Animal> animals = new List<Animal>
{
    new Dog("Рекс", 5, "Овчарка", true),
    new Cat("Мурка", 3, true),
    new Bird("Кеша", 2, 25.5),
    new Fish("Немо", 1, "Солёная"),
    new Dog("Барсик", 4, "Лабрадор", false),
    new Cat("Снежок", 2, false)
};

// ==================================================================
// 1. ДЕЛЕГАТЫ (Delegates)
// ==================================================================
Console.WriteLine("=== 1. Пример работы с ДЕЛЕГАТАМИ ===\n");

AnimalDelegate printName = (animal) => Console.WriteLine($"Имя животного: {animal.Name}");

foreach (var animal in animals)
{
    printName(animal);
}

Console.WriteLine("\nВызов метода через несколько делегатов:");
AnimalDelegate multiHandler = null;
foreach (var animal in animals.Take(3))
{
    multiHandler += LogToConsole;
}

if (multiHandler != null)
{
    multiHandler(animals[0]);
    multiHandler(animals[1]);
    multiHandler(animals[2]);
}

// ==================================================================
// 2. ЛЯМБДА-ВЫРАЖЕНИЯ (Lambda Expressions)
// ==================================================================
Console.WriteLine("\n\n=== 2. Пример работы с ЛЯМБДА-ВЫРАЖЕНИЯМИ ===\n");

// Фильтрация только собак
var dogs = animals.OfType<Dog>();
Console.WriteLine("Только собаки:");
foreach (var dog in dogs)
{
    dog.DisplayInfo();
    dog.Fetch();
    Console.WriteLine("-------------------");
}

// Фильтрация по условию (возраст меньше 4 лет)
var youngAnimals = animals.Where(a => a.Age < 4);
Console.WriteLine("\nМолодые животные (возраст < 4):");
foreach (var animal in youngAnimals)
{
    animal.DisplayInfo();
}

// ИСПРАВЛЕНИЕ ОШИБКИ CS1061:
// Используем OfType<Cat>() для фильтрации и приведения типа к Cat.
// Теперь переменная cat в цикле имеет тип Cat, и метод Purr() доступен.
var indoorCats = animals.OfType<Cat>().Where(cat => cat.IsIndoor);
Console.WriteLine("\nДомашние коты:");
foreach (var cat in indoorCats)
{
    Console.Write($"{cat.Name} ");
    cat.Purr(); // Теперь ошибка устранена
}

// Сортировка по возрасту
var sortedByAge = animals.OrderBy(a => a.Age);
Console.WriteLine("\nСортировка по возрасту (от молодого к старому):");
foreach (var animal in sortedByAge)
{
    Console.WriteLine($"  {animal.Name}: {animal.Age} лет");
}

// Агрегация данных
double averageAge = animals.Average(a => a.Age);
Console.WriteLine($"\nСредний возраст животных: {averageAge:F2} года");

// ==================================================================
// 3. СОБЫТИЯ (Events)
// ==================================================================
Console.WriteLine("\n\n=== 3. Пример работы с СОБЫТИЯМИ ===\n");

foreach (var animal in animals)
{
    animal.AnimalChanged += HandleAnimalUpdate;

    if (animal is Dog dog)
    {
        animal.AnimalChanged += (msg) =>
            Console.WriteLine($"[ДОПОЛНИТЕЛЬНО] Собака '{dog.Name}': {msg}");
    }
}

Console.WriteLine("\nИзменяем возраст животных...\n");

animals[0].UpdateAge(6);   // Рекс
animals[1].UpdateAge(4);   // Мурка
animals[2].UpdateAge(3);   // Кеша
animals[3].UpdateAge(2);   // Немо

Console.WriteLine("\n=== Финальный список всех животных ===\n");
foreach (var animal in animals)
{
    animal.DisplayInfo();
    animal.MakeSound();
    Console.WriteLine(new string('-', 50));
}

Console.WriteLine("\n=== Специфичные действия ===");
foreach (var animal in animals)
{
    switch (animal)
    {
        case Dog dog:
            dog.Fetch();
            break;
        case Cat cat when cat.IsIndoor:
            cat.Purr();
            break;
        case Bird bird:
            bird.Fly();
            break;
        case Fish fish:
            fish.Swim();
            break;
    }
}

// Объявление делегата
public delegate void AnimalDelegate(Animal animal);

// Статические обработчики
static void HandleAnimalUpdate(string message)
{
    Console.WriteLine($"[СОБЫТИЕ] {message}");
}

static void LogToConsole(Animal animal)
{
    Console.WriteLine($"--- Лог через делегат: ---");
    animal.MakeSound();
}

// --- КЛАССЫ ---

public abstract class Animal
{
    public string Name { get; set; }
    public int Age { get; set; }

    public Animal(string name, int age)
    {
        Name = name;
        Age = age;
    }

    public delegate void AnimalEventHandler(string message);
    public event AnimalEventHandler AnimalChanged;

    public void UpdateAge(int newAge)
    {
        if (newAge > 0 && newAge != Age)
        {
            Age = newAge;
            AnimalChanged?.Invoke($"Животное '{Name}' теперь {Age} лет!");
        }
    }

    public virtual void DisplayInfo()
    {
        Console.WriteLine($"Имя: {Name}, Возраст: {Age} лет");
    }

    public abstract void MakeSound();
}

public class Dog : Animal
{
    public string Breed { get; set; }
    public bool IsTrained { get; set; }

    public Dog(string name, int age, string breed, bool isTrained)
        : base(name, age)
    {
        Breed = breed;
        IsTrained = isTrained;
    }

    public override void MakeSound()
    {
        Console.WriteLine($"{Name} лает: Гав-гав!");
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Порода: {Breed}, Дрессирован: {IsTrained}");
    }

    public void Fetch()
    {
        Console.WriteLine($"{Name} приносит мячик!");
    }
}

public class Cat : Animal
{
    public bool IsIndoor { get; set; }

    public Cat(string name, int age, bool isIndoor)
        : base(name, age)
    {
        IsIndoor = isIndoor;
    }

    public override void MakeSound()
    {
        Console.WriteLine($"{Name} мяукает: Мяу-мяу!");
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Домашний: {(IsIndoor ? "Да" : "Нет")}");
    }

    public void Purr()
    {
        Console.WriteLine($"{Name} мурлычет...");
    }
}

public class Bird : Animal
{
    public double Wingspan { get; set; }

    public Bird(string name, int age, double wingspan)
        : base(name, age)
    {
        Wingspan = wingspan;
    }

    public override void MakeSound()
    {
        Console.WriteLine($"{Name} чирикает: Чик-чирик!");
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Размах крыльев: {Wingspan} см");
    }

    public void Fly()
    {
        Console.WriteLine($"{Name} летает!");
    }
}

public class Fish : Animal
{
    public string WaterType { get; set; }

    public Fish(string name, int age, string waterType)
        : base(name, age)
    {
        WaterType = waterType;
    }

    public override void MakeSound()
    {
        Console.WriteLine($"{Name} булькает: Буль-буль!");
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Тип воды: {WaterType}");
    }

    public void Swim()
    {
        Console.WriteLine($"{Name} плавает!");
    }
}

=== 1. Пример работы с ДЕЛЕГАТАМИ ===

Имя животного: Рекс
Имя животного: Мурка
Имя животного: Кеша
Имя животного: Немо
Имя животного: Барсик
Имя животного: Снежок

Вызов метода через несколько делегатов:
--- Лог через делегат: ---
Рекс лает: Гав-гав!
--- Лог через делегат: ---
Рекс лает: Гав-гав!
--- Лог через делегат: ---
Рекс лает: Гав-гав!
--- Лог через делегат: ---
Мурка мяукает: Мяу-мяу!
--- Лог через делегат: ---
Мурка мяукает: Мяу-мяу!
--- Лог через делегат: ---
Мурка мяукает: Мяу-мяу!
--- Лог через делегат: ---
Кеша чирикает: Чик-чирик!
--- Лог через делегат: ---
Кеша чирикает: Чик-чирик!
--- Лог через делегат: ---
Кеша чирикает: Чик-чирик!


=== 2. Пример работы с ЛЯМБДА-ВЫРАЖЕНИЯМИ ===

Только собаки:
Имя: Рекс, Возраст: 5 лет
Порода: Овчарка, Дрессирован: True
Рекс приносит мячик!
-------------------
Имя: Барсик, Возраст: 4 лет
Порода: Лабрадор, Дрессирован: False
Барсик приносит мячик!
-------------------

Молодые животные (возраст < 4):
Имя: Мурка, Возраст: 3 лет
Домашний